# 05 · Modelos con solo el histórico de ventas (configuración A)

**Fase CRISP-DM:** modelado.

**Objetivo:** comprobar si algún modelo supera al estacional ingenuo usando únicamente variables derivadas de las ventas y del calendario (sin fuentes externas: esas son la configuración B). Se comparan los seis modelos del catálogo en cuatro variantes:

- **modo:** un modelo `conjunto` para todos los segmentos, o uno `por_segmento`;
- **escala del target:** `toneladas`, o `relativa` al nivel del último año cerrado.

Todos se evalúan con el mismo walk-forward (horizonte 2) y sobre los mismos meses que las referencias. Los hiperparámetros son los del catálogo; el ajuste se prueba en el notebook 06.

**Regla de privacidad:** solo usa la serie agregada por mes y segmento.

In [ ]:
import time
import warnings

import exp_a
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from sklearn.exceptions import ConvergenceWarning

from lojagas.comun import lakehouse

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", message="IProgress not found")
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 250)
exp_a.estilo_graficos()

H = exp_a.HORIZONTE
RAIZ = exp_a.raiz_lakehouse()
MODOS = ("conjunto", "por_segmento")
ESCALAS = ("toneladas", "relativa")

serie = exp_a.cargar_serie()
print(f"variables sin fuga temporal: {exp_a.verificar_sin_fuga(serie)} meses comprobados")
datos = exp_a.tabla_modelado(serie)
SEGMENTOS = sorted(datos["segmento"].unique())
for escala in ESCALAS:
    print(f"variables en escala {escala}: {exp_a.variables(escala)}")

## 1. Walk-forward de todas las configuraciones

Las predicciones de cada mes de prueba se guardan en el lakehouse de exploración: el notebook 06 parte de ahí, y la configuración B se podrá comparar contra ellas partición por partición.

In [ ]:
inicio = time.time()
partes = [exp_a.evaluar_referencias(datos)]
for nombre in exp_a.MODELOS:
    for modo in MODOS:
        for escala in ESCALAS:
            partes.append(exp_a.evaluar(nombre, datos, modo=modo, escala=escala))
pred = pd.concat(partes, ignore_index=True)
assert pred["pred"].notna().all()

version = lakehouse.escribir(pred, RAIZ, "predicciones", exp_a.TABLA_PREDICCIONES, modo="overwrite")
pruebas = pred["mes"].drop_duplicates()
print(
    f"{pred.groupby(exp_a.CLAVES).ngroups} configuraciones en {time.time() - inicio:.0f} s | "
    f"{len(pruebas)} meses de prueba (de {pruebas.min():%Y-%m} a {pruebas.max():%Y-%m}) | "
    f"versión Delta {version}"
)

## 2. Comparación con el estacional ingenuo

WAPE por segmento. `macro` es el promedio de los segmentos (todos pesan igual) y `TOTAL` es el error sobre la suma de los segmentos (pesa el volumen). La mejora es la reducción del WAPE frente al estacional ingenuo: positiva si el modelo se equivoca menos.

In [ ]:
tabla = exp_a.comparar(pred)
columnas = [*SEGMENTOS, "macro", "TOTAL"]
print("WAPE (%)")
display(tabla[columnas].round(2))
print("Mejora sobre el estacional ingenuo (%)")
display(tabla[[f"mejora_{c}" for c in columnas] + ["segmentos_que_mejoran"]].round(1))

In [ ]:
modelos = tabla[tabla.index.get_level_values("modo") != "referencia"]
ganan = (modelos[[f"mejora_{c}" for c in columnas]] > 0).sum().rename("configuraciones_que_mejoran")
print(f"De {len(modelos)} configuraciones, superan al estacional ingenuo:")
display(ganan.rename(index=lambda c: c.removeprefix("mejora_")).to_frame().T)
print("Configuraciones que mejoran en todos los segmentos a la vez:")
print(int((modelos["segmentos_que_mejoran"] == len(SEGMENTOS)).sum()))

Las tres mejores configuraciones de cada segmento, frente a las dos varas: el estacional ingenuo y el estacional ajustado por días de venta.

In [ ]:
AJUSTADA = ("estacional_dias", "referencia", "toneladas")
filas = []
for columna in [*SEGMENTOS, "TOTAL"]:
    for (modelo, modo, escala), wape in modelos[columna].nsmallest(3).items():
        filas.append(
            {
                "segmento": columna,
                "configuracion": f"{modelo} · {modo} · {escala}",
                "wape": wape,
                "wape_estacional": tabla.loc[exp_a.REFERENCIA, columna],
                "wape_estacional_dias": tabla.loc[AJUSTADA, columna],
                "mejora_sobre_estacional_%": tabla.loc[(modelo, modo, escala), f"mejora_{columna}"],
            }
        )
display(pd.DataFrame(filas).set_index(["segmento", "configuracion"]).round(2))

In [ ]:
orden = tabla.sort_values("macro", ascending=False)
etiquetas = [
    modelo.replace("_", " ") if modo == "referencia" else f"{modelo} · {modo} · {escala}"
    for modelo, modo, escala in orden.index
]
es_referencia = orden.index.get_level_values("modo") == "referencia"
fig, ejes = plt.subplots(1, 2, figsize=(11, 0.27 * len(orden) + 1.2), sharey=True)
for eje, columna in zip(ejes, ("macro", "TOTAL"), strict=True):
    colores = [exp_a.TINTA_SECUNDARIA if r else exp_a.AZUL for r in es_referencia]
    vara = tabla.loc[exp_a.REFERENCIA, columna]
    tope = 2.5 * vara  # las configuraciones muy malas no deben aplastar la escala
    eje.barh(etiquetas, orden[columna].clip(upper=tope), height=0.55, color=colores)
    for posicion, valor in enumerate(orden[columna]):
        if valor > tope:
            eje.text(tope, posicion, f" {valor:.0f}", va="center", color=exp_a.TINTA_SECUNDARIA)
    eje.axvline(vara, color=exp_a.NARANJA, lw=2)
    eje.set_xlim(0, tope * 1.08)
    eje.set_title(f"WAPE {columna} (%)")
    eje.grid(axis="x")
    eje.grid(axis="y", visible=False)
marcas = [
    plt.Line2D([], [], color=exp_a.AZUL, lw=6, label="modelo"),
    plt.Line2D([], [], color=exp_a.TINTA_SECUNDARIA, lw=6, label="referencia sin aprendizaje"),
    plt.Line2D([], [], color=exp_a.NARANJA, lw=2, label="estacional ingenuo (la vara)"),
]
fig.legend(handles=marcas, loc="lower center", ncols=3, bbox_to_anchor=(0.5, -0.02))
plt.tight_layout(rect=(0, 0.04, 1, 1))
plt.show()

## 3. ¿Qué decisión de diseño pesa más?

Promedio del WAPE de las configuraciones según el modo, la escala y la familia de modelo.

In [ ]:
print("Por modo y escala")
display(modelos.groupby(["modo", "escala"])[["macro", "TOTAL"]].mean().round(2))
print("Por familia de modelo (media y mejor variante)")
display(modelos.groupby("modelo")[["macro", "TOTAL"]].agg(["mean", "min"]).round(2))

## 4. ¿Sufren los modelos cuando hay poca historia?

Las primeras particiones entrenan con 12 a 17 meses; las últimas, con 18 o más. Si un modelo solo gana cuando tiene más historia, su ventaja crecerá con el tiempo; si solo gana al principio, es sospechosa.

In [ ]:
pocas = exp_a.comparar(pred[pred["n_entrenamiento"] < 18])
muchas = exp_a.comparar(pred[pred["n_entrenamiento"] >= 18])
por_historia = pd.DataFrame(
    {
        "macro_con_12_a_17_meses": pocas["macro"],
        "mejora_con_12_a_17_meses": pocas["mejora_macro"],
        "macro_con_18_o_mas": muchas["macro"],
        "mejora_con_18_o_mas": muchas["mejora_macro"],
    }
)
display(por_historia.loc[tabla.index[:10]].round(2))

## 5. Registro en MLflow

Experimento `exploracion_a`, separado del oficial. Si MLflow no está levantado, se omite: los resultados ya quedaron en el lakehouse.

In [ ]:
_ = exp_a.registrar_mlflow(
    tabla,
    "05_modelos_a",
    {
        "horizonte": H,
        "min_entrenamiento": exp_a.MIN_ENTRENAMIENTO,
        "lags": str(exp_a.LAGS),
        "meses_de_prueba": len(pruebas),
        "delta_version_predicciones": version,
    },
)

## Decisiones

Las cifras quedan en las salidas de este notebook, que no se suben al repositorio.

1. **En el doméstico y en el total, el histórico de ventas alcanza para superar al estacional ingenuo.** Casi todas las configuraciones lo logran, y las mejores superan también al estacional ajustado por días de venta.
2. **En el agroindustrial ninguna configuración lo supera**, y en el industrial solo unas pocas y por poco. Son segmentos chicos y ruidosos, con muy pocos meses para aprender.
3. **No hay una configuración que gane en todos los segmentos.** El método se elige por segmento (notebook 06), y para el agroindustrial la opción por defecto es el propio estacional.
4. **Un modelo por segmento es mejor que uno conjunto.** El conjunto en toneladas con modelos lineales falla en los segmentos chicos: comparte coeficientes entre series con dos órdenes de magnitud de diferencia.
5. **Los modelos lineales regularizados rinden igual o mejor que los árboles** con tan pocos datos. Es un argumento contra la LSTM: tiene más parámetros que observaciones.
6. **Dos correcciones para el catálogo de `ml/`:** el `alpha` de Lasso y Elastic Net no significa lo mismo en toneladas que en escala relativa si no se estandariza el target, y LightGBM debe fijarse a un hilo (`n_jobs=1`).